# Visualização das bancas de defesa e qualificação do ICMC-USP

**SCC0252 – Visualização Computacional – 2026/2**
**Prof. Jean R. Ponciano**
ICMC-USP São Carlos

**Projeto de Exploração e Análise Visual de Dados – Entrega 1**

| Nome | Nº USP |
|------|--------|
| Gabriel da Costa Merlin | 12544420 |
| Guilherme Antonio Costa Bandeira | 14575620 |
| João Pedro Mori Machado | 13671831 |

## 0. Apresentação

### 0.1 Motivação

As bancas de qualificação e de defesa do ICMC-USP são divulgadas por e-mail, em avisos semanais que informam aluno, orientador, título, data e local. Cada aviso mostra o que acontece naquela semana, mas ninguém enxerga o conjunto. Como alunos de graduação que pretendem ingressar na pós-graduação, temos pouca noção de como esse processo funciona na prática, e um conjunto de dados estruturado com visualizações interativas pode dar essa noção.

O interesse vai além do nosso:
- **Para quem pretende ingressar na pós ou se aproxima da própria banca:** saber quando as bancas costumam acontecer, como são as do programa de interesse (mestrado ou doutorado, presencial ou remota) e quais temas cada orientador acompanha ajuda a escolher programa e orientador e a planejar o cronograma.
- **Para o ICMC:** saber em que semanas as bancas se concentram e quantas são remotas ajuda a dimensionar salas, equipamentos de transmissão e apoio técnico.
- **Para os professores:** entender a carga de bancas em cada época e os temas que circulam em cada programa ajuda a se preparar para as bancas de seus orientandos e as de convidados.

### 0.2 Contextualização do problema

O ICMC-USP São Carlos abriga cinco programas de pós-graduação: Ciências de Computação e Matemática Computacional (PPG-CCMC), Estatística (PIPGES), Matemática (PPG-Mat), Matemática, Estatística e Computação Aplicadas à Indústria (MECAI) e Matemática em Rede Nacional (PROFMAT). Em cada um, o aluno passa por bancas de qualificação e de defesa, no mestrado ou no doutorado, de forma presencial ou remota.

Essas informações existem, mas não estão organizadas para análise: chegam em e-mails de texto livre, e nenhuma tabela mostra como o volume de bancas varia ao longo do ano ou quais combinações de programa, nível, tipo e modalidade são comuns ou raras. O tema de cada orientador, em particular, não é um campo do aviso, e fica escondido no título do trabalho, que mistura português, inglês e vocabulário técnico de cada área.

Este trabalho transforma esses avisos em um conjunto de dados estruturado e o explora com visualizações interativas, em torno de três questões: quando as bancas acontecem, que perfis de banca existem e quais são raros, e se os orientadores se concentram em poucos temas ou cobrem áreas variadas. As seções seguintes descrevem o conjunto de dados, as questões e as visualizações.

## 1. Introdução

### 1.1 Descrição do dataset

O dataset contém **1150 bancas** (defesas e qualificações de mestrado e doutorado) de **5 programas de pós-graduação** do ICMC, com **14 variáveis** por banca, guardadas em `data/eventos.csv`. Cada linha é uma banca. As bancas ocorrem entre 15/03/2022 e 29/10/2026, e os e-mails de origem foram enviados entre 11/03/2022 e 23/10/2026. São **930 alunos** e **162 orientadores** distintos.

Os programas são:
- Ciências de Computação e Matemática Computacional - PPG-CCMC (766 bancas)
- Estatística - PIPGES (165)
- Matemática - PPG-Mat (115)
- Matemática, Estatística e Computação Aplicadas à Indústria - MECAI (65)
- Matemática em Rede Nacional - PROFMAT (39)

**Dicionário de dados**

| Coluna | Descrição | Tipo de variável |
|---|---|---|
| `id` | Identificador da banca (hash de aluno + tipo + nível) | Categórico |
| `tipo` | Qualificação ou Defesa | Ordinal (a qualificação vem antes da defesa) |
| `nivel` | Mestrado ou Doutorado | Ordinal (o doutorado vem depois do mestrado) |
| `programa` | Programa de pós-graduação, com a sigla no final (5 valores) | Categórico |
| `titulo` | Título do trabalho | Categórico (texto livre) |
| `aluno` | Nome do(a) aluno(a) | Categórico |
| `orientador` | Nome do(a) orientador(a), com grafia padronizada | Categórico |
| `data` | Data da banca (AAAA-MM-DD) | Quantitativo, sequencial, com semântica temporal |
| `dia_semana` | Dia da semana da banca (0 = segunda-feira, 4 = sexta-feira) | Ordinal, cíclico, com semântica temporal |
| `hora` | Horário da banca (HH:MM), vazio se o e-mail não informou | Quantitativo, cíclico, com semântica temporal |
| `local` | Código da sala (se `icmc` = 1), texto do e-mail (se `icmc` = 0) ou vazio | Categórico, com semântica espacial |
| `icmc` | 1 = sala do ICMC, 0 = outro lugar, vazio = sem local registrado (bancas remotas e uma presencial) | Categórico (dois valores e vazio) |
| `remoto` | 1 = remota, 0 = presencial | Categórico (dois valores) |
| `atualizado` | 1 se a versão final veio de e-mail de correção | Categórico (dois valores) |
| `tema` | Último nível da árvore de temas do programa, atribuído por LLM a partir do título ("Outros" se nenhum serve) | Categórico, hierárquico, derivado do título |
| `macro_tema` | Primeiro nível da árvore de temas do programa, obtido da árvore a partir do tema | Categórico, hierárquico, derivado do título |

As duas últimas linhas, `tema` e `macro_tema`, não fazem parte do dataset. São atributos derivados do `titulo` e usados só na Q3. Como a classificação por LLM é demorada, o resultado fica guardado em `question3/temas_llm.csv`, um arquivo auxiliar ligado ao dataset pelo `id`. Ele tem as colunas `id`, `programa`, `titulo`, `tema` e `macro_tema`, e as três primeiras repetem o dataset.

**Como ler a classificação.** Os tipos seguem a Aula 02. Um atributo é *categórico* (sem ordem), *ordinal* (com ordem, mas sem aritmética) ou *quantitativo* (com ordem e aritmética). Nos atributos ordenados, a direção é *sequencial* (do menor ao maior) ou *cíclica* (volta ao início, como os dias da semana). *Semântica temporal ou espacial* indica que o atributo descreve tempo ou lugar, *hierárquico* que os valores se organizam em níveis (macro-tema e tema) e *derivado* que o atributo foi calculado a partir de outro. O `tipo` e o `nivel` são ordinais pela natureza dos dados, mas nas visualizações das Q1 e Q2 são tratados como categorias, sem usar a ordem.

**Particularidades dos dados**

O dataset é gerado por um script de extração sobre e-mails de texto livre, e não representa com 100% de fidelidade o conteúdo dos e-mails. As particularidades conhecidas são:

- **Cobertura temporal:** 2022 (a partir de março) e 2026 (até outubro) são anos incompletos, o que deve ser considerado ao comparar anos. Entre março de 2022 e outubro de 2026, todos os meses têm ao menos uma banca.
- **Horário:** `hora` está vazia em uma única banca (a qualificação de doutorado de Bruno Estanislau Holtz, em 13/08/2026), porque o e-mail não informou o horário.
- **Local das bancas remotas:** `local` e `icmc` ficam vazios nas bancas remotas (731 de 1150). Uma banca presencial também está sem local registrado (defesa de Misael Costa Junior, em 14/06/2024).
- **Locais fora do ICMC:** 16 bancas estão com `icmc` = 0, sendo 13 na UFSCar, 1 na FFCLRP (USP Ribeirão Preto), 1 defesa em sessão fechada (Maria Lydia Fioravanti, 29/05/2026), que não é remota nem tem sala, e 1 banca que foi classificada como fora do ICMC por um problema na extração: o local aparece como "Sala 3002 do ICMC-USP", sem hífen no código da sala.
- **Textos:** títulos e nomes seguem o que está no e-mail, então podem conter erros de digitação e cortes. Ao menos 3 títulos terminam de forma abrupta (em "de", em "com" e em vírgula).
- **Cancelamentos:** bancas canceladas sem aviso por e-mail não são detectadas.

In [1]:
# Setup: imports das três questões, caminhos dos dados e configuração geral.
# As paletas acessíveis (daltonismo) são definidas junto do código de cada questão.

import html
import math
import unicodedata
import warnings
import zlib

import ipywidgets as widgets
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, clear_output, display
from plotly.subplots import make_subplots

# Caminhos relativos à pasta viz/, onde este notebook está
CAMINHO = "../data/eventos.csv"                    # dataset gerado por data/scrape_emails.py
ARQ_TEMAS = "question3/temas_llm.csv"         # tema de cada banca, gerado por viz/question3/classificar_temas.py


### 1.2 Questões de investigação

Com os dados descritos acima, vamos responder três questões.

**Q1. Quando as bancas acontecem?**

Como o volume de bancas varia ao longo do ano, e esse padrão se repete entre os anos e difere entre programas, tipos (defesa ou qualificação) e níveis (mestrado ou doutorado)?

**Q2. Que perfis de banca existem e quais são raros?**

Como as defesas se distribuem entre os programas, entre mestrado e doutorado e entre bancas remotas e presenciais, e quais combinações dominam ou são raras?

**Q3. Os orientadores são especialistas ou generalistas?**

Os orientandos de cada professor se concentram em poucos temas ou cobrem uma gama de áreas?

#### 1.2.1 O conjunto tem dados para respondê-las?

Uma análise rápida mostra que sim: cada questão tem variação suficiente para ser explorada.

In [2]:
# Análise preliminar: o conjunto tem variação suficiente para cada questão (poucas linhas por questão)
df_pre = pd.read_csv(CAMINHO)
df_pre["data"] = pd.to_datetime(df_pre["data"])
df_pre["sigla"] = df_pre["programa"].str.split(" - ").str[-1]

# Q1: bancas por mês, somando os anos
print("Q1. Bancas por mês (todos os anos somados)")
por_mes = df_pre["data"].dt.month.value_counts().sort_index()
por_mes.index = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
display(por_mes.rename("bancas").to_frame().T)

# Q2: defesas e qualificações, nível e % remoto por programa
print("\nQ2. Perfil de cada programa")
q2 = pd.DataFrame({
    "defesas": df_pre[df_pre["tipo"] == "Defesa"].groupby("sigla").size(),
    "qualificações": df_pre[df_pre["tipo"] == "Qualificação"].groupby("sigla").size(),
    "mestrado": df_pre[df_pre["nivel"] == "Mestrado"].groupby("sigla").size(),
    "doutorado": df_pre[df_pre["nivel"] == "Doutorado"].groupby("sigla").size(),
    "% remoto": (df_pre.groupby("sigla")["remoto"].mean() * 100).round(1),
}).fillna({"defesas": 0, "qualificações": 0, "mestrado": 0, "doutorado": 0})
q2[["defesas", "qualificações", "mestrado", "doutorado"]] = q2[["defesas", "qualificações", "mestrado", "doutorado"]].astype(int)
display(q2)

# Q3: bancas por orientador
print("\nQ3. Bancas por orientador")
por_orient = df_pre["orientador"].value_counts()
display(pd.DataFrame({
    "orientadores": [len(por_orient)],
    "mediana de bancas": [por_orient.median()],
    "máximo": [por_orient.max()],
    "% nos 10 mais ativos": [round(por_orient.head(10).sum() / por_orient.sum() * 100, 1)],
}))

Q1. Bancas por mês (todos os anos somados)


,jan,fev,mar,abr,mai,jun,jul,ago,set,out,nov,dez
bancas,18,68,101,125,158,107,59,91,103,172,68,80



Q2. Perfil de cada programa


,defesas,qualificações,mestrado,doutorado,% remoto
sigla,,,,,
MECAI,65,0,65,0,84.6
PIPGES,80,85,84,81,75.8
PPG-CCMC,343,423,451,315,64.2
PPG-Mat,115,0,63,52,21.7
PROFMAT,39,0,39,0,87.2



Q3. Bancas por orientador


,orientadores,mediana de bancas,máximo,% nos 10 mais ativos
0,162,5.0,28,18.8


Somando os anos, as bancas se distribuem de forma desigual pelo ano, com 172 em outubro e 158 em maio, contra 18 em janeiro, o que leva à Q1. Os programas também diferem entre si: o PPG-Mat, o MECAI e o PROFMAT não têm nenhuma qualificação no conjunto, porque elas não são divulgadas na lista, e o MECAI e o PROFMAT só têm bancas de mestrado. A modalidade varia bastante entre programas, com 21,7% de bancas remotas no PPG-Mat e 87,2% no PROFMAT, o que leva à Q2, restrita às defesas por esse motivo. Por fim, os orientadores são desiguais: dos 162 orientadores, a mediana é de 5 bancas e o máximo é de 28, e os dez mais ativos respondem por 18,8% das bancas, o que leva à Q3.

## 2. Pré-processamento

Os dados vêm da agenda semanal de defesas e qualificações do ICMC-USP São Carlos, enviada por e-mail. Exportamos esses e-mails do Gmail da USP pelo Google Takeout, em formato `.mbox` (busca por `subject:"[grad-l]" "Qualificações"`; 259 mensagens, em 240 conversas do Gmail, enviadas entre 11/03/2022 e 23/10/2026), e o script `data/scrape_emails.py` gera o `data/eventos.csv`, com uma banca por linha. Nada disso roda neste notebook: ele parte do CSV pronto.

O script:
- **Extrai** de cada e-mail, com expressões regulares, tipo, nível, programa, título, aluno(a), orientador(a), data, hora e local. E-mails em HTML são convertidos em texto, e os trechos citados de respostas anteriores são removidos.
- **Limpa e padroniza** as irregularidades do texto livre (linhas quebradas, negrito com asteriscos, assinaturas e rodapés colados), os cinco programas (nome com a sigla no final) e as grafias dos orientadores. O ano quase nunca aparece na data, então é inferido pela data de envio do e-mail.
- **Deduplica:** a mesma banca aparece em vários e-mails (agenda semanal, "Atualizado", "Retificação"), então cada banca recebe um `id` (hash de aluno + tipo + nível) e fica a versão do e-mail mais recente. Foram lidas 1398 ocorrências, que resultaram em 1150 bancas.
- **Deriva** `dia_semana`, `remoto`, `icmc` e `atualizado` (ver o dicionário na seção 1.1).

Os campos que não puderem ser extraídos ficam vazios (nas bancas remotas, `local` e `icmc` ficam vazios por definição). O pré-processamento específico de cada questão (colunas auxiliares, filtros e a classificação de temas) está no início da seção dela.

## 3. Análise visual

Cada questão segue a mesma ordem: pergunta, pré-processamento específico, código das visualizações e insights.

### 3.1 Q1: Quando as bancas acontecem?

Como o volume de bancas varia ao longo do ano, e esse padrão se repete entre os anos e difere entre programas, tipos (defesa ou qualificação) e níveis (mestrado ou doutorado)?

#### 3.1.1 Pré-processamento (Q1)

**Variáveis usadas:** `data` (de onde saem a semana e o ano ISO), `programa`, `tipo` e `nivel` (filtros e painéis). `aluno` e `hora` aparecem apenas nas dicas (tooltips).

O `eventos.csv` não tem colunas de semana nem de ano, então a Q1 deriva, sem gravá-las no CSV:
- `segunda`: a segunda-feira da semana da banca;
- `sigla`: só a sigla do programa;
- o ano e a semana ISO de cada semana, que dão o eixo X (semana do ano) e a série de cada linha (um ano).

**Semanas sem dados.** Semanas sem e-mails não viram zero: sequências de 6 ou mais semanas sem nenhuma banca são marcadas como "sem dados" (a linha se interrompe e uma faixa cinza marca o intervalo). Semanas com e-mails, mas sem bancas da categoria filtrada, são zero de fato. Anos que não cobrem as semanas 1 a 52 aparecem como "(parcial)" na legenda (2022 e 2026).

In [3]:
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
CORES_ANO = ["#56B4E9", "#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
SIMBOLOS_ANO = ["circle", "square", "diamond", "triangle-up", "x", "cross"]
CINZA = "#c4c8cf"
MAX_LINHAS_TOOLTIP = 10
_SEMANAS = list(range(1, 54))
_TICKS = [int(pd.Timestamp(2025, m, 1).isocalendar().week) for m in range(1, 13)]


def carregar(caminho_csv):
    df = pd.read_csv(caminho_csv, parse_dates=["data"]).dropna(subset=["data"]).copy()
    df["segunda"] = df["data"] - pd.to_timedelta(df["data"].dt.weekday, unit="D")
    df["sigla"] = df["programa"].str.rsplit(" - ", n=1).str[-1]
    return df


def montar_grade(df, gap_min=6, manual=()):
    """Uma linha por semana (segunda-feira) entre o 1º e o último evento.
    `gap` = sequência de >= gap_min semanas seguidas sem nenhuma banca (= sem e-mails, não zero).
    Recesso curto (dez/jan) continua zero. `manual`: [(inicio, fim), ...] força "sem dados".
    Calculado no df COMPLETO, antes dos filtros."""
    seg = pd.date_range(df["segunda"].min(), df["segunda"].max(), freq="7D")
    vazio = df.groupby("segunda").size().reindex(seg, fill_value=0).to_numpy() == 0
    gap = np.zeros(len(seg), bool)
    i = 0
    while i < len(seg):
        if vazio[i]:
            j = i
            while j < len(seg) and vazio[j]:
                j += 1
            if j - i >= gap_min:
                gap[i:j] = True
            i = j
        else:
            i += 1
    iso = seg.isocalendar()
    g = pd.DataFrame({"ano": iso["year"].astype(int).to_numpy(),
                      "semana": iso["week"].astype(int).to_numpy(), "gap": gap}, index=seg)
    for ini, fim in manual:
        g.loc[pd.Timestamp(ini):pd.Timestamp(fim), "gap"] = True
    return g

In [4]:
def _lista(d):
    if d is None or len(d) == 0:
        return ""
    d = d.sort_values(["data", "hora"], na_position="last")
    linhas = []
    for r in d.itertuples():
        h = f" {r.hora}" if isinstance(r.hora, str) else ""
        linhas.append(f"{r.data:%d/%m}{h} · {r.tipo}/{r.nivel} · {html.escape(str(r.aluno))} ({r.sigla})")
    extra = len(linhas) - MAX_LINHAS_TOOLTIP
    linhas = linhas[:MAX_LINHAS_TOOLTIP]
    if extra > 0:
        linhas.append(f"… e mais {extra}")
    return "<br>" + "<br>".join(linhas)


def _semanal(df, g, periodo=None):
    """Gera matrizes temporais de contagem e textos de tooltip de forma 100% segura.

    - Se a semana está num gap global do ano (sem e-mails), o valor é NaN (a linha quebra).
    - Se a semana está ativa, mas a categoria não possui eventos, o valor é exatamente 0.0."""
    anos = sorted(g["ano"].unique())
    ia = {a: k for k, a in enumerate(anos)}

    cont = np.full((len(anos), 53), np.nan)
    texto = np.full((len(anos), 53), "", dtype=object)

    ini, fim = periodo if periodo else (g.index[0], g.index[-1])
    grupos = dict(tuple(df.groupby("segunda")))

    for seg, r in g.iterrows():
        a_val = int(r["ano"])
        s_val = int(r["semana"])

        if a_val not in ia or s_val < 1 or s_val > 53:
            continue

        a_idx = ia[a_val]
        s_idx = s_val - 1

        # Verifica se a semana está fora dos limites de início/fim reais registrados para o ano
        limites_ano = g[g["ano"] == a_val].index
        fora_dos_limites = (seg < limites_ano.min()) or (seg > limites_ano.max())

        if r["gap"] or not (ini <= seg <= fim) or fora_dos_limites:
            cont[a_idx, s_idx] = np.nan
            texto[a_idx, s_idx] = f"<b>Semana {s_val}/{a_val}</b> ({seg:%d/%m})<br><i>sem dados de e-mail</i>"
        else:
            # Semana ativa com dados: se houver eventos, conta; se não, é estritamente 0.0
            d = grupos.get(seg)
            n = 0 if d is None else len(d)
            cont[a_idx, s_idx] = float(n)

            intervalo_texto = f"{seg:%d/%m}–{seg + pd.Timedelta(days=6):%d/%m}"
            texto[a_idx, s_idx] = f"<b>Semana {s_val}/{a_val}</b> ({intervalo_texto})<br><b>{n}</b> banca(s) real(is)" + _lista(d)

    return anos, cont, texto


def _suavizar(cont, k):
    """Calcula a média móvel sem deixar pontas flutuantes e respeitando NaNs."""
    if k <= 1:
        return cont
    out = np.full_like(cont, np.nan)
    for i, linha in enumerate(cont):
        s_linha = pd.Series(linha)
        # Roda a média móvel apenas onde temos valores válidos (incluindo os zeros que acabamos de fixar)
        m = s_linha.rolling(k, center=True, min_periods=1).mean().to_numpy()
        out[i] = np.where(np.isnan(linha), np.nan, m)
    return out


def _rotulo_ano(g, a):
    sem = g.loc[g["ano"] == a, "semana"]
    parcial = sem.min() > 1 or sem.max() < 52
    return f"{a} (parcial)" if parcial else str(a)


def _tracos(df, g, periodo, suavizar, media, mostrar_legenda, cor_ano):
    anos, cont, texto = _semanal(df, g, periodo)
    liso = _suavizar(cont, suavizar)
    tr = []
    for k, a in enumerate(anos):
        t = []
        for s_idx, (x, v_suave) in enumerate(zip(texto[k], liso[k])):
            if x == "" or "<i>" in x:
                t.append(x)
            else:
                v_real = cont[k, s_idx]
                if suavizar > 1 and not np.isnan(v_suave):
                    # Torna visualmente indubitável que a altura do ponto é a média móvel e não o valor bruto real
                    txt_suave = (
                        f"<br>-----------------------------"
                        f"<br><b>Valor real desta semana:</b> {int(v_real)} banca(s)"
                        f"<br><b>Média móvel (altura plotada):</b> {v_suave:.2f} banca(s)"
                        f"<br><i>*Suavização de {suavizar} semanas ativa</i>"
                    )
                    # Removemos a menção duplicada para evitar confusão
                    base_text = x.split("<br><b>")[0] + f"<br>({x.split(' (')[1].split(')')[0]})"
                    t.append(base_text + txt_suave)
                else:
                    t.append(x)

        tr.append(go.Scatter(
            x=_SEMANAS, y=liso[k], mode="lines+markers", name=_rotulo_ano(g, a),
            legendgroup=str(a), showlegend=mostrar_legenda,
            line=dict(color=cor_ano[a], width=2, shape="linear"),
            marker=dict(size=7, symbol=SIMBOLOS_ANO[k % len(SIMBOLOS_ANO)]), connectgaps=False,
            text=t, hovertemplate="%{text}<extra></extra>"))

    if media:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            m = np.nanmean(liso, axis=0)
        n = np.isfinite(liso).sum(axis=0)
        m = np.where(n >= 2, m, np.nan)
        tr.append(go.Scatter(
            x=_SEMANAS, y=m, mode="lines", name="média dos anos", legendgroup="media",
            showlegend=mostrar_legenda, line=dict(color="#222", width=2.5, dash="dash"),
            connectgaps=False,
            text=[f"<b>Semana {s}</b><br>média dos anos: {v:.1f} bancas<br>({k} anos com dados)"
                  for s, v, k in zip(_SEMANAS, m, n)],
            hovertemplate="%{text}<extra></extra>"))
    return tr


def _faixas_sem_dados(fig, g, row=None, col=None):
    for a in sorted(g["ano"].unique()):
        sub = g[(g["ano"] == a) & g["gap"]]
        if len(sub):
            x0 = int(sub["semana"].min())
            x1 = int(sub["semana"].max())
            fig.add_vrect(
                x0=x0 - 0.5, x1=x1 + 0.5, fillcolor=CINZA, opacity=0.15,
                layer="below", line_width=0, row=row, col=col
            )


def grafico(df, g, dividir=None, suavizar=1, periodo=None, media=True, y_comum=True, titulo=None):
    """dividir: None, 'programa', 'tipo' ou 'nivel' (um painel por categoria)."""
    cor_ano = {a: CORES_ANO[i % len(CORES_ANO)] for i, a in enumerate(sorted(g["ano"].unique()))}

    if not dividir:
        fig = go.Figure()
        tracos = _tracos(df, g, periodo, suavizar, media, True, cor_ano)
        for t in tracos:
            fig.add_trace(t)
        _faixas_sem_dados(fig, g)

        # Ajuste dinâmico de Y para painel único
        _, cont, _ = _semanal(df, g, periodo)
        liso = _suavizar(cont, suavizar)
        max_val = np.nanmax(liso) if np.any(np.isfinite(liso)) else 1
        teto_y = max(3, math.ceil(max_val * 1.15))

        fig.update_layout(
            title=titulo,
            xaxis=dict(tickmode="array", tickvals=_TICKS, ticktext=MESES, range=[0.5, 53.5], title="Semana do ano"),
            yaxis=dict(range=[0, teto_y], title="Nº de bancas na semana"),
            height=500, margin=dict(l=60, r=40, t=80, b=60), hovermode="closest",
            plot_bgcolor="#f8f9fa", paper_bgcolor="white"
        )
        return fig

    cats = sorted(df[dividir].dropna().unique())
    n_cats = len(cats)
    cols = 1 if n_cats <= 2 else 2
    rows = math.ceil(n_cats / cols)

    fig = make_subplots(
        rows=rows, cols=cols, subplot_titles=cats,
        shared_xaxes=True, shared_yaxes=y_comum,
        vertical_spacing=0.1, horizontal_spacing=0.08
    )

    # Calcular o valor máximo global para ajustar a escala do Y compartilhado ou individual
    max_global = 0.1
    for c in cats:
        sub_df = df[df[dividir] == c]
        _, cont, _ = _semanal(sub_df, g, periodo)
        liso = _suavizar(cont, suavizar)
        if np.any(np.isfinite(liso)):
            max_global = max(max_global, np.nanmax(liso))

    for idx, c in enumerate(cats):
        r_idx = (idx // cols) + 1
        c_idx = (idx % cols) + 1
        sub_df = df[df[dividir] == c]
        mostrar_leg = (idx == 0)
        tracos = _tracos(sub_df, g, periodo, suavizar, media, mostrar_leg, cor_ano)
        for t in tracos:
            fig.add_trace(t, row=r_idx, col=c_idx)
        _faixas_sem_dados(fig, g, row=r_idx, col=c_idx)

    # Determina o teto do eixo Y com folga de 15%
    teto_y = max(3, math.ceil(max_global * 1.15))

    fig.update_layout(
        title=titulo, height=330 * rows, margin=dict(l=60, r=40, t=80, b=60),
        hovermode="closest", plot_bgcolor="#f8f9fa", paper_bgcolor="white"
    )

    for r in range(1, rows + 1):
        for c in range(1, cols + 1):
            fig.update_xaxes(tickmode="array", tickvals=_TICKS, ticktext=MESES, range=[0.5, 53.5], row=r, col=c)
            fig.update_yaxes(range=[0, teto_y] if y_comum else [0, None], row=r, col=c)

    return fig


def painel_q1(caminho_csv, gap_min=6, manual=()):
    """Painel interativo sem opção de suavização. Requer ipywidgets."""
    W = widgets

    df = carregar(caminho_csv)
    g = montar_grade(df, gap_min, manual)

    def checks(titulo, opcoes):
        cbs = [W.Checkbox(value=True, description=str(o), indent=False,
                          layout=W.Layout(width="auto")) for o in opcoes]
        return W.VBox([W.HTML(f"<b>{titulo}</b>"), *cbs]), cbs

    box_p, cb_p = checks("Programa", df["sigla"].value_counts().index)
    box_t, cb_t = checks("Tipo", sorted(df["tipo"].unique()))
    box_n, cb_n = checks("Nível", sorted(df["nivel"].unique()))
    dividir = W.ToggleButtons(options=[("nenhum", None), ("programa", "programa"), ("tipo", "tipo"),
                                       ("nível", "nivel")], description="Dividir por")
    media = W.Checkbox(value=True, description="mostrar média dos anos", indent=False)
    ycom = W.Checkbox(value=True, description="mesmo eixo Y nos painéis", indent=False)
    slider = W.SelectionRangeSlider(
        options=[(f"{d:%d/%m/%Y}", d) for d in g.index], index=(0, len(g) - 1),
        description="Período", continuous_update=False,
        layout=W.Layout(width="95%"), style={"description_width": "60px"})

    # Criando container de saída e forçando a renderização do widget
    out = W.Output()

    def atualizar(*_):
        ps = [c.description for c in cb_p if c.value]
        ts = [c.description for c in cb_t if c.value]
        ns = [c.description for c in cb_n if c.value]
        ini, fim = slider.value
        d = df[df["sigla"].isin(ps) & df["tipo"].isin(ts) & df["nivel"].isin(ns)]
        n = d["segunda"].between(ini, fim).sum()
        # Chamar gráfico sem suavização (suavizar=1)
        fig = grafico(d, g, dividir.value, 1, (ini, fim), media.value, ycom.value,
                      titulo=f"Bancas por semana do ano, uma linha por ano · {n} bancas no filtro")
        with out:
            clear_output(wait=True)
            fig.update_layout(width=1100)
            display(fig)

    for w in cb_p + cb_t + cb_n + [slider, dividir, media, ycom]:
        w.observe(atualizar, "value")

    painel_completo = W.VBox([W.HBox([box_p, box_t, box_n, W.VBox([media, ycom])], layout=W.Layout(gap="40px")),
                              dividir, slider, out])

    display(painel_completo)
    atualizar()

#### 3.1.2 Visualização 1: painel interativo

Gráfico com filtros de programa, tipo e nível, divisão em painéis (por programa, tipo ou nível), recorte de período, média dos anos e eixo Y comum ou independente.

In [5]:
painel_q1(CAMINHO)

#### 3.1.3 Insights (Q1)

**Resposta.** Sim, o padrão se repete entre os anos, mas a repetição vem do PPG-CCMC. Os meses mais cheios são maio e outubro, com um pico de defesas em meados de dezembro, e os mais vazios são janeiro e julho. Os programas menores variam bastante de um ano para o outro, e as qualificações são mais sazonais do que as defesas.

**Visão geral e repetição entre os anos.** O padrão se repete entre os anos. Somando as bancas por mês em 2023, 2024 e 2025 (anos completos), os totais mensais de um ano acompanham os dos outros (correlação entre 0,73 e 0,80). Janeiro é o mês com menos bancas nos três anos (4, 5 e 4) e julho o segundo mais vazio (10, 6 e 12). Outubro é o mais movimentado em 2023 e 2025 (43 e 35), e maio em 2024 (34, contra 32 em outubro). A repetição vem sobretudo do PPG-CCMC, que concentra 766 das 1150 bancas (67%). Nos programas menores, o padrão é menos estável: o PIPGES praticamente não se repete de um ano para o outro, e o PPG-Mat só em parte.

**Por programa (painel dividido por programa).**
- **PPG-CCMC:** é o programa com mais bancas, com média de 3,2 por semana, contra 0,2 a 0,7 nos demais, e é o único com padrão sazonal claro. Há alta em maio e junho, queda em julho e agosto, novo pico no fim de outubro e início de novembro e um pico em meados de dezembro. Na virada do ano a linha cai a zero, porque não há bancas nas últimas semanas de dezembro e em parte de janeiro. O maior volume semanal é de 13 bancas, em abril e maio de 2026 (semanas de 27/04, 04/05 e 25/05).
- **PIPGES (Estatística):** é o segundo maior programa. Na maioria das semanas tem 0 ou 1 banca, com picos de 4 a 6 bancas em agosto (o máximo, 6, na semana de 11/08/2025).
- **PPG-Mat:** na maioria das semanas tem 0 ou 1 banca, com mais bancas em fevereiro e março. O maior pico do programa, 8 bancas na semana de 13/02/2023, não se repetiu em nenhum outro ano.
- **MECAI e PROFMAT:** têm poucas bancas (65 e 39), com 0 ou 1 por semana na maior parte do tempo. Semanas com 2 bancas aparecem de vez em quando e, nesses programas, não configuram um pico. O máximo do MECAI é 3 bancas, na semana de 24/02/2025, e o do PROFMAT é 3 bancas, na semana de 07/11/2022.

**Por nível (painel dividido por nível).** Somando todos os programas, as bancas de doutorado variam menos ao longo do ano: ficam em geral entre 2 e 4 por semana e caem para cerca de 1 no meio do ano. As de mestrado oscilam mais: chegam a cerca de 6 por semana em maio e em outubro, caem para cerca de 1 nas férias de meio de ano e ficam em 3 ou 4 no restante.

**Por tipo (painel dividido por tipo).** As defesas são mais constantes ao longo do ano: ficam em cerca de 3 por semana na maior parte do tempo, com um pico no fim de fevereiro e início de março (média de 7 por semana nessa época) e outro em meados de dezembro (cerca de 6). Há uma queda para cerca de 1 em meados de julho. As qualificações são mais sazonais: começam o ano com 0 a 1 por semana e sobem de forma gradual até cerca de 5 no início de maio, caem para cerca de 1 em julho e agosto, voltam a subir até cerca de 6 em outubro e depois caem de novo. Ou seja, as qualificações se concentram em dois períodos do ano e as defesas se distribuem de forma mais uniforme. Esses valores são médias semanais entre os anos, e em semanas como a do pico de março a média depende de poucos anos (4 a 10 defesas por ano).

**Cuidados na leitura.**
- **Programas pequenos:** com poucas bancas por semana, as linhas oscilam entre 0, 1 e 2, e um único pico (como o do PPG-Mat em 2023) não indica padrão. Compare a forma das curvas só nos programas com mais bancas.
- **Anos parciais:** 2022 (a partir de março) e 2026 (até outubro) cobrem só parte do ano, então não podem ser comparados semana a semana com 2023 a 2025.
- **Eixo Y comum:** com o mesmo eixo nos painéis, as curvas dos programas pequenos ficam achatadas perto do zero, e o dominante é o CCMC.

### 3.2 Q2: Que perfis de banca existem e quais são raros?

Como as defesas se distribuem entre os programas, entre mestrado e doutorado e entre bancas remotas e presenciais, e quais combinações dominam ou são raras?

#### 3.2.1 Pré-processamento (Q2)

**Variáveis usadas:** `tipo` (filtro: só entram as defesas), `programa` (reduzido à sigla), `nivel` e `remoto` (convertida em `modalidade`, com os valores "Remoto" e "Presencial"). `programa`, `nivel` e `modalidade` são as três dimensões do treemap.

A Q2 usa apenas as defesas, por dois motivos. Cada defesa representa um aluno que concluiu o curso, então ninguém é contado duas vezes, o que não vale para as qualificações, que se somam às defesas do mesmo aluno. E o PPG-Mat, o MECAI e o PROFMAT não têm nenhuma qualificação no dataset, porque elas não são divulgadas na lista de e-mails desses programas. Incluir qualificações faria o PPG-CCMC e o PIPGES parecerem maiores do que são em relação aos demais e distorceria a comparação entre programas.

A coluna `remoto` é convertida em `modalidade` ("Remoto" ou "Presencial") e o nome do programa é reduzido à sigla, para caber nos rótulos do gráfico.

In [6]:
# Q2: só as defesas, com a modalidade (remoto/presencial) e a sigla do programa
TIPO_DEFESA = 'defesa'
MAPA_MODALIDADE = {1: 'Remoto', 0: 'Presencial'}
SEPARADOR_PROGRAMA = ' - '   # a sigla é o trecho depois do último separador

df_q2 = pd.read_csv(CAMINHO)
df_q2_defesas = df_q2[df_q2['tipo'].str.lower() == TIPO_DEFESA].copy()   # sem as qualificações
df_q2_defesas['modalidade'] = df_q2_defesas['remoto'].map(MAPA_MODALIDADE)
df_q2_defesas['programa'] = df_q2_defesas['programa'].str.rsplit(SEPARADOR_PROGRAMA, n=1).str[-1].str.strip()

#### 3.2.2 Visualização 2: treemap hierárquico

Treemap das defesas por programa, nível e modalidade, com a ordem da hierarquia escolhida pelos seletores. A área de cada caixa é proporcional ao número de defesas.

In [7]:
# Q2: configuração do treemap (seletores, paleta e tamanhos)

# Mapeamento amigável para os nomes dos seletores
OPCOES_DIMENSOES = {
    'Programa de Pós-Graduação': 'programa',
    'Nível (Mestrado/Doutorado)': 'nivel',
    'Modalidade (Remoto/Presencial)': 'modalidade',
}

# Paletas sequenciais, uma para cada categoria do nível externo: do branco até uma cor
# Okabe-Ito e depois uma versão escurecida dela. Cada programa tem a mesma cor das outras
# questões (CCMC azul, PIPGES laranja, PPG-Mat verde). As cinco cores continuam
# distinguíveis nas simulações de deuteranopia, protanopia e tritanopia.
def _escala(cor_hex):
    r, g, b = mcolors.to_rgb(cor_hex)
    return mcolors.LinearSegmentedColormap.from_list(cor_hex, [(1, 1, 1), (r, g, b), (0.45 * r, 0.45 * g, 0.45 * b)])

CMAPS_PROGRAMA = {
    'PPG-CCMC': _escala('#0072B2'),   # azul
    'PIPGES': _escala('#E69F00'),     # laranja
    'PPG-Mat': _escala('#009E73'),    # verde-azulado
    'MECAI': _escala('#CC79A7'),      # rosa
    'PROFMAT': _escala('#F0E442'),    # amarelo
}
# Quando o nível externo não é o programa (nível ou modalidade), as categorias usam as cores nesta ordem
CMAPS_DISPONIVEIS = list(CMAPS_PROGRAMA.values())

# Posição (0 a 1) em cada paleta
COR_EXTERNO = 0.4          # caixa do nível externo
COR_MEDIO = 0.55           # caixa do nível médio (um pouco mais escura)
COR_FOLHA_MIN = 0.3        # folha com a menor quantidade
COR_FOLHA_AMPLITUDE = 0.6  # quanto o tom sobe até a folha com a maior quantidade
COR_FOLHA_UNICA = 0.7      # quando todas as folhas têm a mesma quantidade

LARGURA, ALTURA = 1100, 650
TAMANHO_MIN_TEXTO = 10

In [8]:
# Q2: funções do hover e do treemap

# Rótulos das dimensões, exibidos no hover (as que faltarem usam o nome da coluna)
ROTULOS_DIM = {
    'programa': 'Programa',
    'nivel': 'Nível',
    'modalidade': 'Modalidade',
}
MAX_IRMAOS = 8  # máximo de caixas irmãs com valor listadas no hover
MAX_ZEROS = 5   # máximo de caixas irmãs zeradas listadas no hover


def fmt_pct(x):
  return f'{100 * x:.1f}'.replace('.', ',') + '%'


def fmt_defesas(n):
  return f'{n} defesa' if n == 1 else f'{n} defesas'


def cor_texto(hex_cor):
  """Branco em fundo escuro, quase preto em fundo claro."""
  r, g, b = mcolors.to_rgb(hex_cor)
  return '#1a1a1a' if 0.299 * r + 0.587 * g + 0.114 * b > 0.6 else 'white'


def escurecer(hex_cor, fator=0.6):
  r, g, b = mcolors.to_rgb(hex_cor)
  return mcolors.to_hex((r * fator, g * fator, b * fator))


def montar_hover(path, dims, valor, valor_pai, total, irmaos):
  """Texto HTML do hover de uma caixa.

  path: rótulos do topo até a caixa (ex.: ['PPG-Mat', 'Mestrado', 'Remoto'])
  dims: nomes das dimensões desse mesmo caminho
  valor_pai: quantidade da caixa pai (None no nível externo)
  irmaos: {categoria: quantidade} de TODAS as categorias daquela dimensão,
          inclusive as que valem 0 (e por isso não têm retângulo)
  """
  atual = path[-1]
  linhas = [f'<b>{" › ".join(str(p) for p in path)}</b>']

  # Quantidade exata e porcentagens
  partes = [fmt_defesas(valor)]
  if valor_pai is not None:
    partes.append(f'{fmt_pct(valor / valor_pai)} do grupo')
  partes.append(f'{fmt_pct(valor / total)} do total')
  linhas.append(' · '.join(partes))

  # Posição entre as caixas irmãs e resumo delas (incluindo os zeros)
  if len(irmaos) > 1:
    posicao = 1 + sum(1 for v in irmaos.values() if v > valor)
    empate = sum(1 for v in irmaos.values() if v == valor) > 1
    rotulo = ROTULOS_DIM.get(dims[-1], dims[-1].capitalize())
    linhas.append(
        f'{rotulo} — {posicao}ª maior de {len(irmaos)}'
        + (' (empate)' if empate else '')
    )

    ordenados = sorted(irmaos.items(), key=lambda kv: (-kv[1], str(kv[0])))
    com_valor = [(c, v) for c, v in ordenados if v > 0]
    zeros = [str(c) for c, v in ordenados if v == 0]

    mostrados = com_valor[:MAX_IRMAOS]
    if all(c != atual for c, _ in mostrados):  # garante que a própria caixa apareça
      mostrados = mostrados[:-1] + [(atual, valor)]
    itens = [f'<b>{c}: {v}</b>' if c == atual else f'{c}: {v}' for c, v in mostrados]
    if len(com_valor) > len(mostrados):
      itens.append(f'… +{len(com_valor) - len(mostrados)}')
    linhas.append(' · '.join(itens))

    if zeros:
      txt = ', '.join(zeros[:MAX_ZEROS])
      if len(zeros) > MAX_ZEROS:
        txt += f' … +{len(zeros) - MAX_ZEROS}'
      linhas.append(f'Nenhuma defesa: {txt}')

  return '<br>'.join(linhas)


# Função que atualiza o gráfico dinamicamente com base nas escolhas do usuário
def atualizar_treemap(dim1, dim2, dim3):
  caminho = [dim1, dim2, dim3]

  # Validação para evitar dimensões repetidas na mesma hierarquia
  if len(set(caminho)) < 3:
    print(
        '⚠️ Aviso: Por favor, selecione três dimensões diferentes para cada'
        ' nível da hierarquia.'
    )
    return

  # Agrupamento base com as 3 dimensões
  df_raw = (
      df_q2_defesas
      .groupby([dim1, dim2, dim3])
      .size()
      .reset_index(name='quantidade')
  )

  # Totais e categorias usados no hover (porcentagens, posição e caixas zeradas)
  total_geral = df_raw['quantidade'].sum()
  categorias = {d: sorted(df_raw[d].unique()) for d in caminho}
  tot1 = df_raw.groupby(dim1)['quantidade'].sum().to_dict()
  tot2 = df_raw.groupby([dim1, dim2])['quantidade'].sum().to_dict()
  tot3 = df_raw.set_index([dim1, dim2, dim3])['quantidade'].to_dict()

  # Paleta de cada categoria do nível externo (dim1): por programa, ou pela ordem
  grupos_dim1 = df_raw[dim1].unique()
  mapa_cmaps = {
      g: CMAPS_PROGRAMA.get(g, CMAPS_DISPONIVEIS[i % len(CMAPS_DISPONIVEIS)])
      for i, g in enumerate(grupos_dim1)
  }

  # Listas para construir o DataFrame hierárquico explícito (pais, filhos e nós)
  ids, labels, parents, values, cores_hex, hovers = [], [], [], [], [], []

  # 1. Nível 1: Categoria mais externa (dim1)
  for g in grupos_dim1:
    sub_df = df_raw[df_raw[dim1] == g]
    val_g = sub_df['quantidade'].sum()

    id_1 = str(g)
    ids.append(id_1)
    labels.append(str(g))
    parents.append('')  # Raiz
    values.append(val_g)

    cmap = mapa_cmaps[g]
    # Cor base intermediária para a caixa externa
    cores_hex.append(mcolors.to_hex(cmap(COR_EXTERNO)))
    hovers.append(montar_hover(
        [g], [dim1], val_g, None, total_geral,
        {c: tot1.get(c, 0) for c in categorias[dim1]},
    ))

    # 2. Nível 2: Categoria intermediária (dim2)
    sub_grupos_dim2 = sub_df[dim2].unique()
    for sg2 in sub_grupos_dim2:
      sub2_df = sub_df[sub_df[dim2] == sg2]
      val_sg2 = sub2_df['quantidade'].sum()

      id_2 = f'{id_1} / {sg2}'
      ids.append(id_2)
      labels.append(str(sg2))
      parents.append(id_1)
      values.append(val_sg2)
      # Tom um pouco mais escuro para o nível intermediário
      cores_hex.append(mcolors.to_hex(cmap(COR_MEDIO)))
      hovers.append(montar_hover(
          [g, sg2], [dim1, dim2], val_sg2, val_g, total_geral,
          {c: tot2.get((g, c), 0) for c in categorias[dim2]},
      ))

      # 3. Nível 3: Folhas/Caixas mais internas (dim3) com gradiente baseado na quantidade
      min_val = sub_df['quantidade'].min()
      max_val = sub_df['quantidade'].max()

      for _, row in sub2_df.iterrows():
        sg3 = row[dim3]
        qtd = row['quantidade']

        id_3 = f'{id_2} / {sg3}'
        ids.append(id_3)
        labels.append(str(sg3))
        parents.append(id_2)
        values.append(qtd)

        # Gradiente interno proporcional à quantidade
        if max_val == min_val:
          norm_val = COR_FOLHA_UNICA
        else:
          norm_val = COR_FOLHA_MIN + COR_FOLHA_AMPLITUDE * (qtd - min_val) / (max_val - min_val)
        cores_hex.append(mcolors.to_hex(cmap(norm_val)))
        hovers.append(montar_hover(
            [g, sg2, sg3], caminho, qtd, val_sg2, total_geral,
            {c: tot3.get((g, sg2, c), 0) for c in categorias[dim3]},
        ))

  # Montando o DataFrame final com a estrutura completa da árvore
  df_tree = pd.DataFrame({
      'ids': ids,
      'labels': labels,
      'parents': parents,
      'values': values,
      'cor_hex': cores_hex,
      'hover': hovers,
      'cor_texto': [cor_texto(c) for c in cores_hex],
      'cor_borda': [escurecer(c) for c in cores_hex],
  })

  dicionario_cores = {c: c for c in df_tree['cor_hex'].unique()}

  # Criando o Treemap interativo
  fig = px.treemap(
      df_tree,
      ids='ids',
      names='labels',
      parents='parents',
      values='values',
      branchvalues='total',
      color='cor_hex',
      color_discrete_map=dicionario_cores,
      custom_data=['hover'],
      title=(
          '<b>Defesas do ICMC por perfil de banca</b><br><sub>Hierarquia atual:'
          f' {" -> ".join(caminho)} | área = número de defesas</sub>'
      ),
  )

  # Hover com fundo na cor da própria caixa; o <extra> remove a caixinha secundária
  fig.update_traces(
      hovertemplate='%{customdata[0]}<extra></extra>',
      hoverlabel=dict(
          bgcolor=df_tree['cor_hex'].tolist(),
          bordercolor=df_tree['cor_borda'].tolist(),
          font=dict(size=14, color=df_tree['cor_texto'].tolist()),
          align='left',
      ),
  )

  fig.update_layout(
      margin=dict(t=70, l=20, r=20, b=20),
      width=LARGURA,
      height=ALTURA,
      uniformtext=dict(minsize=TAMANHO_MIN_TEXTO, mode='hide'),
  )
  fig.show()

In [9]:
# Q2: painel interativo (ordem da hierarquia escolhida pelos seletores)

# Criando os dropdowns para o usuário escolher a ordem de aninhamento
q2_dropdown_nivel1 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='programa', description='1º (Externo):'
)
q2_dropdown_nivel2 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='nivel', description='2º (Médio):'
)
q2_dropdown_nivel3 = widgets.Dropdown(
    options=OPCOES_DIMENSOES, value='modalidade', description='3º (Interno):'
)

# Organizando a interface visual com widgets do Jupyter
q2_ui = widgets.VBox(
    [
        widgets.HTML(
            '<b>Escolha a ordem de visualização (todas as caixas herdarão a'
            ' harmonia de cor do primeiro nível):</b>'
        ),
        widgets.HBox([q2_dropdown_nivel1, q2_dropdown_nivel2, q2_dropdown_nivel3]),
    ]
)

# Conectando os seletores à função de atualização
q2_out = widgets.interactive_output(
    atualizar_treemap,
    {'dim1': q2_dropdown_nivel1, 'dim2': q2_dropdown_nivel2, 'dim3': q2_dropdown_nivel3},
)

display(q2_ui, q2_out)


Output()

#### 3.2.3 Insights (Q2)

**Resposta.** O PPG-CCMC domina o conjunto, com 343 das 642 defesas (53,4%), e o perfil mais comum é o do seu mestrado remoto, com 125 defesas (19,5% do total). A modalidade diferencia bastante os programas: o PROFMAT, o MECAI e o PIPGES são quase todos remotos, e o PPG-Mat é o único majoritariamente presencial. Os perfis mais raros são o mestrado presencial do PROFMAT (5 defesas), o doutorado presencial do PIPGES (7) e o mestrado remoto do PPG-Mat (9), e o MECAI e o PROFMAT só têm defesas de mestrado.

Os números abaixo valem para as 642 defesas do dataset, de 640 alunos (dois têm uma defesa de mestrado e outra de doutorado).

**Os programas.** Depois do PPG-CCMC vêm o PPG-Mat (115 defesas, 17,9%), o PIPGES (80, 12,5%), o MECAI (65, 10,1%) e o PROFMAT (39, 6,1%). No total há mais defesas de mestrado (384, 59,8%) do que de doutorado (258), mas isso muda de um programa para outro: o PPG-CCMC (180 de mestrado e 163 de doutorado) e o PPG-Mat (63 e 52) são quase equilibrados, o PIPGES é o único em que o doutorado supera o mestrado (43 contra 37), e o MECAI e o PROFMAT só têm mestrado.

**Remoto ou presencial.** Das 642 defesas, 401 (62,5%) foram remotas. O PROFMAT (87,2% remoto), o MECAI (84,6%) e o PIPGES (78,8%) ficam bem acima dessa média, e o PPG-CCMC (65,3%) fica perto dela, com o mestrado (69,4%) mais remoto do que o doutorado (60,7%). O PPG-Mat (21,7% remoto) é a exceção: 36 das 52 defesas de doutorado (69,2%) e 54 das 63 de mestrado (85,7%) foram presenciais.

**Perfis dominantes e raros.** Depois do mestrado remoto do PPG-CCMC (125 defesas), o segundo maior perfil é o doutorado remoto do mesmo programa (99). No outro extremo ficam o mestrado presencial do PROFMAT (5), o doutorado presencial do PIPGES (7) e o mestrado remoto do PPG-Mat (9). Das 20 combinações possíveis de programa, nível e modalidade, 16 existem, e as 4 que faltam são o doutorado, remoto e presencial, do MECAI e do PROFMAT.

**Trocando a hierarquia.** Com a modalidade no nível externo, o peso dos programas muda. Das 401 defesas remotas, 224 (55,9%) são do PPG-CCMC e só 25 (6,2%) do PPG-Mat. Entre as 241 presenciais, o PPG-CCMC tem 119 (49,4%) e o PPG-Mat 90 (37,3%): o PPG-Mat é pequeno no total, mas pesa bastante nas defesas presenciais.

### 3.3 Q3: Os orientadores são especialistas ou generalistas?

Os orientandos de cada professor se concentram em poucos temas ou cobrem uma gama de áreas?

#### 3.3.1 Pré-processamento (Q3)

**Variáveis usadas:** `orientador`, `programa`, `titulo` (de onde sai o tema) e `id` (liga cada banca ao arquivo de temas). Os derivados `tema` e `macro_tema` ficam em um arquivo auxiliar, `temas_llm.csv`, gerado a partir do título (ver abaixo). A concentração de cada orientador é calculada no notebook a partir deles.

##### 3.3.1.1 Processamento dos títulos: como o tema de cada banca foi obtido

Os avisos não trazem o assunto do trabalho como um campo, ele só aparece de forma implícita no título, em texto livre. Por isso o **tema** (e o **macro-tema**) não vem dos e-mails: foi derivado do título em uma etapa separada, e o notebook apenas carrega o resultado (`question3/temas_llm.csv`).

**Árvores de temas.** Usamos a árvore de temas oficial de cada programa, publicada nas páginas do PPG-CCMC, do PPG-Mat e do PIPGES (guardadas em `question3/temas_existentes.json`). Elas são hierárquicas, mas cada programa as organiza de um jeito: no PPG-Mat há dois níveis (uma área e, dentro dela, os temas), no PIPGES a lista é plana e no PPG-CCMC a estrutura varia por área. Para tratar os três igualmente, usamos só as pontas de cada ramo: o **tema** é o último nível da árvore e o **macro-tema** é o primeiro. Uma área sem sub-temas é, ao mesmo tempo, macro-tema e tema.

**Classificação.** O script `question3/classificar_temas.py` usa um modelo de linguagem executado localmente (gemma4:12b, via Ollama), o que evita custo por requisição e deixa o procedimento reproduzível. O LLM **não roda no notebook**. Para cada banca, o modelo recebe só o programa e o título, nunca o nome do orientador nem do aluno, de modo que o orientador não influencia o tema atribuído. O pedido lista os caminhos completos da árvore para dar contexto, mas manda escolher sempre o último nome de cada caminho. A temperatura é zero (a mesma entrada dá a mesma saída) e a resposta é restrita por um esquema JSON, que só aceita um dos temas do programa da banca ou "Outros", usado quando nenhum tema serve de fato. Assim o modelo nunca devolve um tema fora da lista, e o macro-tema não é decidido por ele: sai da árvore, a partir do tema escolhido.

**Quem ficou de fora.** MECAI e PROFMAT (104 bancas) não têm árvore de temas equivalente nas fontes que usamos, e criar uma seria uma decisão nossa, não uma referência oficial. Por isso eles aparecem apenas nas Q1 e Q2. Das 1046 bancas dos três programas classificados, 41 foram para "Outros" e saem da Q3, restando 1005 bancas de 136 orientadores.

| Programa | Macro-temas | Temas | Bancas | "Outros" |
|---|---|---|---|---|
| PPG-CCMC | 7 | 18 | 766 | 24 |
| PPG-Mat | 3 | 22 | 115 | 6 |
| PIPGES | 4 | 4 | 165 | 11 |
| **Total** | **14** | **44** | **1046** | **41** |

Macro-temas e temas são os da árvore de cada programa; nem todos os temas aparecem nas bancas (por exemplo, o PPG-Mat usa 8 dos 22).

**Limites.** Conferimos manualmente 50 bancas sorteadas ao acaso e os temas pareceram coerentes com os títulos. Como não existem rótulos oficiais de tema para as bancas, não foi possível calcular uma taxa de acerto: os temas devem ser lidos como uma aproximação do assunto de cada trabalho. Cada banca recebe um único tema, o que simplifica a análise e ignora trabalhos que cruzam duas áreas.

In [10]:
# Q3: constantes e junção das bancas com o tema de cada uma

ESPACO = HTML('<div style="height:48px"></div>')      # espaço entre os controles e o gráfico
ROTULO = {"tema": "Tema", "macro_tema": "Macro-tema"}      # nome de cada nível nos controles, hovers e títulos
COR = {"PPG-CCMC": "#0072B2", "PIPGES": "#E69F00", "PPG-Mat": "#009E73"}   # paleta que funciona com daltonismo

eventos = pd.read_csv(CAMINHO)[["id", "orientador"]]
temas = pd.read_csv(ARQ_TEMAS)
bancas = temas.merge(eventos, on="id")
n_outros = (bancas.tema == "Outros").sum()
bancas = bancas[bancas.tema != "Outros"]

PROGRAMAS = [p for p in COR if p in set(bancas.programa)]     # na ordem da paleta
MAX_BANCAS = int(bancas.orientador.value_counts().max())      # teto do slider "Mín. bancas": o orientador com mais bancas


##### 3.3.1.2 Métrica de especialização

**Métrica.** Para cada orientador, a **concentração** é a fração das suas bancas que cai no tema mais frequente dele (o tema principal). Um orientador com todas as bancas em um só tema tem concentração de 100% (especialista), e quanto menor o valor, mais espalhadas estão as bancas (generalista). A mesma conta vale para os macro-temas. Essa é a medida de especialização usada nos dois gráficos.

In [11]:
# Q3: medida de concentração por orientador e painel de controles

def sem_acento(s):
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode().lower()

def casa(nome, busca):
    return busca.strip() != "" and sem_acento(busca.strip()) in sem_acento(nome)

def curto(s, n=22):
    return s if len(s) <= n else s[: n - 1] + "…"

def perfis(programa, minimo, nivel):
    """Uma linha por orientador (perfil) e a tabela orientador x tema com o nº de bancas (conta)."""
    b = bancas if programa == "Todos" else bancas[bancas.programa == programa]
    conta = pd.crosstab(b.orientador, b[nivel])
    n = conta.sum(axis=1)
    conta, n = conta[n >= minimo], n[n >= minimo]
    p = conta.div(n, axis=0)
    perfil = pd.DataFrame({
        "n": n,
        "tema_principal": conta.idxmax(axis=1),
        "concentracao": p.max(axis=1),
    })
    perfil["programa"] = b.groupby("orientador").programa.agg(lambda s: s.value_counts().index[0]).reindex(perfil.index)
    return perfil, conta


def painel_q3(grafico, **controles):
    """Controles + gráfico numa caixa só. Parecido com o interact, mas a cada mudança a saída é
    limpa NA HORA antes de redesenhar (o interact limpa só quando chega a próxima saída)."""
    saida = widgets.Output(layout=widgets.Layout(width="100%"))     # ocupa a largura toda da saída

    def redesenhar(*_):
        saida.clear_output()
        with saida:
            grafico(**{nome: c.value for nome, c in controles.items()})

    for c in controles.values():
        c.observe(redesenhar, names="value")
    display(widgets.VBox([*controles.values(), saida], layout=widgets.Layout(width="100%")))
    redesenhar()


#### 3.3.2 Visualização 3.A: dispersão por orientador

Um ponto por orientador: número de bancas no eixo horizontal e concentração no tema principal no vertical, com cor por programa. Filtros de programa, nível (tema ou macro-tema), mínimo de bancas e busca por nome.

In [12]:
def grafico_dispersao(programa="Todos", nivel="tema", minimo=4, busca=""):
    rot = ROTULO[nivel]
    perfil, _ = perfis(programa, minimo, nivel)
    perfil = perfil.assign(x=[n + ((zlib.crc32(o.encode()) % 1000) / 1000 - 0.5) * 0.36
                              for o, n in zip(perfil.index, perfil.n)])
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas")

    fig = go.Figure()
    for prog in PROGRAMAS:
        q = perfil[perfil.programa == prog]
        if q.empty:
            continue
        texto = [f"<b>{o}</b> ({r.n} bancas)<br>{rot} principal: {r.tema_principal}"
                 f"<br>Concentração no {rot.lower()} principal: {r.concentracao:.0%} ({round(r.concentracao * r.n)}/{r.n})" for o, r in q.iterrows()]
        fig.add_trace(go.Scatter(x=q.x, y=q.concentracao, mode="markers", name=prog, text=texto, hoverinfo="text",
                                 marker=dict(color=COR[prog], size=10, opacity=0.85, line=dict(color="white", width=1))))

    a, b = (perfil.n.min(), perfil.n.max()) if len(perfil) else (1, MAX_BANCAS)

    achados = perfil[[casa(o, busca) for o in perfil.index]]
    if busca.strip():
        print(f"{len(achados)} encontrado(s) (circulado)" if len(achados) else "Nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    if len(achados):
        fig.add_trace(go.Scatter(x=achados.x, y=achados.concentracao, mode="markers+text", text=list(achados.index),
                                 textposition="top center", showlegend=False, hoverinfo="skip",
                                 marker=dict(size=20, color="rgba(0,0,0,0)", line=dict(color="black", width=2))))

    fig.update_xaxes(title_text="Nº de bancas do orientador", title_font=dict(size=12), range=[a - 0.7, b + 0.7], dtick=1, zeroline=False)
    fig.update_yaxes(title_text=f"Concentração no {rot.lower()} principal (% das bancas)", title_font=dict(size=12), range=[0, 1.04], tickformat=".0%",
                     zeroline=False)
    fig.update_layout(height=470, margin=dict(l=70, r=20, t=10, b=120), plot_bgcolor="#fafafa",
                      legend=dict(orientation="h", y=-0.28, x=0))
    fig.show(config={"responsive": True, "displayModeBar": False})        # acompanha a largura da saída; sem a barra do plotly


painel_q3(grafico_dispersao,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="Programa"),
       nivel=widgets.Dropdown(options=[(r, k) for k, r in ROTULO.items()], description="Nível"),
       minimo=widgets.IntSlider(value=min(4, MAX_BANCAS), min=1, max=MAX_BANCAS, description="Mín. bancas"),
       busca=widgets.Text(placeholder="Digite um nome", description="Buscar"))


#### 3.3.3 Visualização 3.B: mapa de calor orientador × tema

Cada coluna é um orientador e cada linha um tema, com a cor indicando a fração das bancas do orientador naquele tema (cinza = nenhuma banca). Mesmos filtros da 3.A, mais a ordem das colunas.

In [13]:
ESCALA = [[0, "#e5e7eb"], [0.00001, "#e5e7eb"], [0.0000101, "#c6dbef"], [0.25, "#6baed6"],
          [0.5, "#3182bd"], [0.75, "#08519c"], [1, "#08306b"]]       # zero = cinza; resto, azul claro -> escuro

def grafico_heatmap(programa="Todos", nivel="tema", minimo=4, ordem="esp", busca=""):
    rot = ROTULO[nivel]
    perfil, conta = perfis(programa, minimo, nivel)
    perfil = {
        "esp": perfil.sort_values(["concentracao", "n"], ascending=False),
        "ger": perfil.sort_values(["concentracao", "n"], ascending=[True, False]),
        "n": perfil.sort_values("n", ascending=False),
        "nome": perfil.sort_index(),
    }[ordem]
    conta = conta.loc[perfil.index]
    conta = conta.loc[:, conta.sum() > 0]                     # só as colunas (temas) que alguém usou
    pct = conta.div(perfil.n, axis=0)
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas")

    z = pct.values.T                                                   # linhas = temas, colunas = orientadores
    texto = [[f"<b>{o}</b> ({perfil.n[o]} bancas)<br>{rot}: {t}<br>" + (f"Concentração: {pct.loc[o, t]:.0%} das bancas ({conta.loc[o, t]}/{perfil.n[o]})"
                                                                      if conta.loc[o, t] else "Nenhuma banca nesse tema")
              for o in conta.index] for t in conta.columns]
    fig = go.Figure(go.Heatmap(x=list(conta.index), y=list(conta.columns), z=z,
                               text=texto, hoverinfo="text", colorscale=ESCALA, zmin=0,
                               zmax=1, xgap=0, ygap=0,
                               colorbar=dict(title="% das bancas",
                                             tickformat=".0%",
                                             x=1.01, xanchor="left", y=1, yanchor="top",
                                             lenmode="pixels", len=260, thickness=14)))
    idx = [i for i, o in enumerate(conta.index) if casa(o, busca)]
    if busca.strip():
        print(f"{len(idx)} encontrado(s) (coluna com contorno preto)" if idx else "Nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    for i in idx:
        fig.add_shape(type="rect", xref="x", yref="y", x0=i - 0.5, x1=i + 0.5, y0=-0.5, y1=len(conta.columns) - 0.5,
                      line=dict(color="black", width=2.5), layer="above")
    topo = int(0.82 * 5.5 * max((len(o) for o in conta.index), default=0)) + 40      # altura do maior nome girado em 55 graus
    fig.update_xaxes(tickangle=-55, side="top", automargin=True, tickfont=dict(size=9), type="category", fixedrange=True)
    fig.update_yaxes(autorange="reversed", automargin=True, tickfont=dict(size=9), fixedrange=True, tickvals=list(conta.columns),
                     ticktext=[curto(t) for t in conta.columns])
    fig.update_layout(height=max(380, topo + 26 * len(conta.columns) + 40), margin=dict(l=130, r=20, t=topo, b=40), dragmode=False,
                      plot_bgcolor="white")
    fig.show(config={"responsive": True, "displayModeBar": False})        # acompanha a largura da saída


painel_q3(grafico_heatmap,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="Programa"),
       nivel=widgets.Dropdown(options=[(r, k) for k, r in ROTULO.items()], description="Nível"),
       minimo=widgets.IntSlider(value=min(4, MAX_BANCAS), min=1, max=MAX_BANCAS, description="Mín. bancas"),
       ordem=widgets.Dropdown(options=[("Mais especialistas primeiro", "esp"), ("Mais generalistas primeiro", "ger"),
                                       ("Nº de bancas", "n"), ("Nome", "nome")], description="Ordem"),
       busca=widgets.Text(placeholder="Digite um nome", description="Buscar"))


#### 3.3.4 Insights (Q3)

**Resposta.** Para a maioria, nenhuma das duas coisas. A concentração mediana no tema principal é de 61%; só 10 dos 93 orientadores têm todas as bancas em um único tema e 33 têm metade ou menos delas no tema principal. Há especialistas claros (nove, todos do PPG-CCMC) e generalistas claros (três), e a resposta depende da escala: em macro-temas, a concentração mediana sobe para 75%.

Os números abaixo valem para a visão padrão (mínimo de 4 bancas, nível tema): são 93 dos 136 orientadores.

**A maioria não é especialista nem generalista puro.** A concentração mediana no tema principal é de 61%. Só 10 orientadores têm todas as bancas em um único tema, e 33 têm metade ou menos delas no tema principal.

**A concentração cai com o número de bancas.** Entre os 23 orientadores com 4 ou 5 bancas a mediana é de 80%, e entre os 17 com 15 bancas ou mais ela é de 53%. Só um deles (Processamento de Linguagem Natural) tem todas as bancas no mesmo tema. É por isso que o mínimo de bancas importa: com poucas bancas, um valor alto pode ser coincidência.

**Programas.** A concentração mediana é de 75% no PPG-Mat (15 orientadores), 58% no PIPGES (18) e 57% no PPG-CCMC (60). A comparação pede cuidado, porque os programas têm números diferentes de temas na árvore (22 no PPG-Mat, 18 no PPG-CCMC e 4 no PIPGES): com poucos temas, concentrar-se é mais fácil.

**Tema ou macro-tema.** Trocando o nível para macro-tema, a mediana sobe para 75% e o número de orientadores com todas as bancas em um só macro-tema passa de 10 para 23 (no PPG-CCMC, de 4 para 15). Muitos orientadores têm uma grande área bem definida, mas variam entre os temas dentro dela.

**Casos extremos.**
- **Especialistas:** nove orientadores, todos do PPG-CCMC, têm pelo menos 8 bancas e 80% ou mais delas em um só tema: Aprendizado de Máquina e Mineração de Dados (quatro deles), Mecânica dos Fluidos Computacional (dois), Computação Aplicada à Educação (dois) e Processamento de Linguagem Natural (um).
- **Generalistas:** três orientadores, também do PPG-CCMC, têm de 8 a 20 bancas e 35% ou menos no tema principal. São os generalistas mais claros.

**Temas disseminados e de nicho (3.B).** Aprendizado de Máquina e Mineração de Dados é o tema mais presente, com bancas de 47 dos 93 orientadores, seguido de Processamento de Linguagem Natural (26) e Inferência Estatística (17). Essas linhas do mapa de calor aparecem bem preenchidas, o que sugere temas transversais. Temas de nicho têm poucas células coloridas, como Análise Harmônica e Teoria dos Números, com um orientador cada. A mediana é de 3 temas por orientador (metade deles entre 2 e 4): 10 atuam em um só tema e 13 em cinco ou mais, no máximo seis.

**Cuidados na leitura.**
- **Temas atribuídos por LLM:** vêm do título e não foram validados com rótulos oficiais; títulos curtos ou genéricos podem cair no tema errado.
- **Um tema por banca:** trabalhos que misturam áreas são reduzidos a um tema, e a especialização medida é a das árvores oficiais, não do assunto real.
- **Temas diferentes por programa:** o número de temas varia (22, 18 e 4), o que torna frágil a comparação entre programas. O PPG-CCMC reúne 60 dos 93 orientadores.
- **Aprendizado de máquina em duas linhas:** Aprendizado de Máquina e Mineração de Dados (PPG-CCMC) e Aprendizado de Máquina (PIPGES) vêm de árvores diferentes e aparecem como linhas separadas, apesar de tratarem de assuntos próximos.
- **Qualificações e defesas juntas:** a contagem mistura os dois tipos, então o mesmo orientando pode aparecer duas vezes. Ficam de fora MECAI, PROFMAT e as bancas "Outros".

## 4. Conclusão

As respostas às três questões estão nos *Insights* de cada seção. A discussão crítica e as limitações, as lições aprendidas, os próximos passos e as referências estão no relatório em PDF.